In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install -q dagshub mlflow
import mlflow
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")
dagshub_username = user_secrets.get_secret("DAGSHUB_USERNAME")

os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_username
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token

mlflow.set_tracking_uri(f"https://dagshub.com/{dagshub_username}/ML_assignment2.mlflow")
mlflow.end_run()

print(f"Successfully connected to {dagshub_username}'s MLflow server!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 83.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Data Loading & Cleaning

In [4]:
import gc

train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

del train_transaction, train_identity
gc.collect()

mlflow.end_run()
mlflow.set_experiment("XGBoost_Training")
with mlflow.start_run(run_name="XGBoost_Cleaning"):
    null_percent = train.isnull().sum() / len(train)
    cols_to_drop = null_percent[null_percent > 0.90].index.tolist()
    train.drop(columns=cols_to_drop, inplace=True)
    
    mlflow.log_param("missing_threshold", 0.90)
    mlflow.log_metric("cols_dropped", len(cols_to_drop))
    print(f"Cleaning complete. Dropped {len(cols_to_drop)} columns.")

Cleaning complete. Dropped 12 columns.
🏃 View run XGBoost_Cleaning at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/91326afcf2ef466586c4e9153c202dcd
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


## Feature Engineering

In [5]:
mlflow.end_run()
with mlflow.start_run(run_name="XGBoost_Feature_Engineering"):
    train['Transaction_hour'] = (train['TransactionDT'] // 3600) % 24
    train['Transaction_day'] = (train['TransactionDT'] // (3600 * 24)) % 7
    train['card1_amt_mean'] = train.groupby(['card1'])['TransactionAmt'].transform('mean')
    train['amt_to_mean_card1'] = train['TransactionAmt'] / train['card1_amt_mean']
    
    mlflow.log_param("added_features", ["hour", "day", "amt_to_mean_card1"])
    mlflow.log_metric("total_features_after_eng", len(train.columns))
    
    cat_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'addr1', 'addr2']
    cat_features = [c for c in cat_features if c in train.columns]
    
    mlflow.log_param("time_strat", "cyclical_extraction")
    print(f"Feature engineering complete.")

Feature engineering complete.
🏃 View run XGBoost_Feature_Engineering at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/520c2383f967490fb5a6119dabb00f8f
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


## Feature Selection

In [6]:
mlflow.end_run()
with mlflow.start_run(run_name="XGBoost_Feature_Selection"):
    cols_to_drop = ['isFraud', 'TransactionID', 'TransactionDT']
    constant_cols = [col for col in train.columns if train[col].nunique() <= 1]
    all_to_drop = list(set(cols_to_drop + constant_cols))
    
    X = train.drop(columns=cols_to_drop)
    y = train['isFraud']
    
    mlflow.log_param("dropped_identifiers", str(cols_to_drop))
    mlflow.log_param("dropped_constant_cols", str(constant_cols))
    mlflow.log_metric("final_feature_count", X.shape[1])
    print(f"Final feature count for training: {X.shape[1]}")

Final feature count for training: 423
🏃 View run XGBoost_Feature_Selection at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/49b020f744954378b5851d03ab78b0f1
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


## Training and Model Pipeline

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from xgboost import XGBClassifier

manual_cat_cols = ['card1', 'card2', 'card3', 'card5', 'addr1', 'addr2']
auto_cat_cols = X.select_dtypes(include=['object']).columns.tolist()
cat_cols = list(set(manual_cat_cols + auto_cat_cols))
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

mlflow.end_run()
with mlflow.start_run(run_name="XGBoost_baseline"):
    preprocessor = ColumnTransformer(transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', 'passthrough', num_cols)
    ])
    
    model = XGBClassifier(
        n_estimators=500, 
        max_depth=6, 
        learning_rate=0.1, 
        tree_method='hist',      
        random_state=42
    )

    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    clf.fit(X_train, y_train)

    from sklearn.metrics import roc_auc_score
    val_preds = clf.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_preds)
    
    mlflow.log_params({
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "hist"
    })
    mlflow.log_metric("val_auc", auc_score)
    mlflow.sklearn.log_model(clf, "model", registered_model_name="XGB_Fraud_Model")
    
    print(f"[XGBoost_baseline] Validation AUC: {auc_score:.4f}")

2026/05/01 18:43:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:43:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'XGB_Fraud_Model' already exists. Creating a new version of this model...
2026/05/01 18:43:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_Fraud_Model, version 5
Created version '5' of model 'XGB_Fraud_Model'.


[XGBoost_baseline] Validation AUC: 0.9532
🏃 View run XGBoost_baseline at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/2523f0593015473a99f4c03237df208e
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


In [8]:
from sklearn.metrics import roc_auc_score

def build_xgb_preprocessor(num_cols, cat_cols):
    return ColumnTransformer(transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
        ('num', 'passthrough', num_cols)
    ])

experiments = [
    {
        "run_name": "XGB_deeper",
        "model": XGBClassifier(
            n_estimators=500,
            max_depth=8,
            learning_rate=0.1,
            tree_method='hist',
            random_state=42
        )
    },
    {
        "run_name": "XGB_slower_lr",
        "model": XGBClassifier(
            n_estimators=1000,
            max_depth=6,
            learning_rate=0.05,
            tree_method='hist',
            random_state=42
        )
    },
    {
        "run_name": "XGB_regularized",
        "model": XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.1,
            tree_method='hist',
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.5,
            random_state=42
        )
    },
    {
        "run_name": "XGB_scale_pos",
        "model": XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.1,
            tree_method='hist',
            scale_pos_weight=29,
            random_state=42
        )
    },
]

for exp in experiments:
    mlflow.end_run()
    with mlflow.start_run(run_name=exp["run_name"]):
        clf = Pipeline(steps=[
            ('preprocessor', build_xgb_preprocessor(num_cols, cat_cols)),
            ('classifier', exp["model"])
        ])

        clf.fit(X_train, y_train)
        val_preds = clf.predict_proba(X_val)[:, 1]
        auc_score = roc_auc_score(y_val, val_preds)

        mlflow.log_params(exp["model"].get_params())
        mlflow.log_metric("val_auc", auc_score)
        mlflow.sklearn.log_model(clf, "model", registered_model_name=exp["run_name"])

        print(f"[{exp['run_name']}] Validation AUC: {auc_score:.4f}")

2026/05/01 18:57:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:57:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_deeper'.
2026/05/01 18:57:40 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_deeper, version 1
Created version '1' of model 'XGB_deeper'.


[XGB_deeper] Validation AUC: 0.9669
🏃 View run XGB_deeper at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/a3805aa62b9d4a18b53eede095f4c190
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


2026/05/01 18:59:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:59:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_slower_lr'.
2026/05/01 19:00:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_slower_lr, version 1
Created version '1' of model 'XGB_slower_lr'.


[XGB_slower_lr] Validation AUC: 0.9527
🏃 View run XGB_slower_lr at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/5c5a0577a74b4526ac60ea98879af044
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


2026/05/01 19:01:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 19:01:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_regularized'.
2026/05/01 19:01:36 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_regularized, version 1
Created version '1' of model 'XGB_regularized'.


[XGB_regularized] Validation AUC: 0.9562
🏃 View run XGB_regularized at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/301258dc078f42bf8fc24c20061d7e88
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


2026/05/01 19:02:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 19:03:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_scale_pos'.
2026/05/01 19:03:06 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_scale_pos, version 1
Created version '1' of model 'XGB_scale_pos'.


[XGB_scale_pos] Validation AUC: 0.9565
🏃 View run XGB_scale_pos at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/199ddee0c5b2466fae6b8eee8241365b
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0


### Hypermarameter Tuning

In [10]:
from sklearn.model_selection import RandomizedSearchCV

mlflow.end_run()
with mlflow.start_run(run_name="XGB_hyperparameter_tuning"):

    clf = Pipeline(steps=[
        ('preprocessor', build_xgb_preprocessor(num_cols, cat_cols)),
        ('classifier', XGBClassifier(tree_method='hist', random_state=42))
    ])

    param_dist = {
        'classifier__n_estimators': [300, 500, 700],
        'classifier__max_depth': [7, 8, 9, 10],      
        'classifier__learning_rate': [0.05, 0.1, 0.2],
        'classifier__subsample': [0.8, 0.9, 1.0],
        'classifier__colsample_bytree': [0.7, 0.8, 1.0],
        'classifier__reg_alpha': [0, 0.1, 0.5],
        'classifier__reg_lambda': [1, 1.5, 2]
    }

    X_tune, _, y_tune, _ = train_test_split(
        X_train, y_train, train_size=0.3, random_state=42, stratify=y_train
    )

    search = RandomizedSearchCV(
        clf,
        param_distributions=param_dist,
        n_iter=10,
        scoring='roc_auc',
        cv=2,
        n_jobs=1,
        random_state=42,
        verbose=2
    )

    search.fit(X_tune, y_tune)

    best_clf = search.best_estimator_
    best_clf.fit(X_train, y_train)

    val_preds = best_clf.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_preds)

    mlflow.log_params(search.best_params_)
    mlflow.log_metric("best_cv_auc", search.best_score_)
    mlflow.log_metric("val_auc", auc_score)
    mlflow.sklearn.log_model(best_clf, "model", registered_model_name="XGB_tuned")

    print(f"Best params: {search.best_params_}")
    print(f"Best CV AUC: {search.best_score_:.4f}")
    print(f"Final Validation AUC: {auc_score:.4f}")

Fitting 2 folds for each of 10 candidates, totalling 20 fits
[CV] END classifier__colsample_bytree=0.7, classifier__learning_rate=0.2, classifier__max_depth=9, classifier__n_estimators=500, classifier__reg_alpha=0.5, classifier__reg_lambda=1.5, classifier__subsample=1.0; total time=  29.9s
[CV] END classifier__colsample_bytree=0.7, classifier__learning_rate=0.2, classifier__max_depth=9, classifier__n_estimators=500, classifier__reg_alpha=0.5, classifier__reg_lambda=1.5, classifier__subsample=1.0; total time=  30.6s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=10, classifier__n_estimators=700, classifier__reg_alpha=0.5, classifier__reg_lambda=2, classifier__subsample=0.9; total time=  55.8s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=10, classifier__n_estimators=700, classifier__reg_alpha=0.5, classifier__reg_lambda=2, classifier__subsample=0.9; total time=  55.6s
[CV] END classifier

2026/05/01 19:58:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 19:58:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'XGB_tuned'.
2026/05/01 19:59:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB_tuned, version 1
Created version '1' of model 'XGB_tuned'.


Best params: {'classifier__subsample': 0.9, 'classifier__reg_lambda': 2, 'classifier__reg_alpha': 0.5, 'classifier__n_estimators': 700, 'classifier__max_depth': 10, 'classifier__learning_rate': 0.05, 'classifier__colsample_bytree': 0.8}
Best CV AUC: 0.9233
Final Validation AUC: 0.9723
🏃 View run XGB_hyperparameter_tuning at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0/runs/5c9d9154b11b4aa5bd65812555021a29
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/0
